In [1]:
import os
from openai import OpenAI
import rich

In [2]:
API_KEY = os.environ.get('OPENAI_API_KEY')
BASE_URL = os.environ.get('OPENAI_BASE_URL')
MODEL = "gpt-5.4"

openai = OpenAI(api_key=API_KEY, base_url=BASE_URL)

**Using system prompt to generate structured output**

# Chat Completion API

https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat

In [3]:
system_prompt = """
Extract the event information from the users message and provide output in
exact json format as follows -
{'name': 'Dinner', 'date': 'Monday', 'participants': ['Michael', 'David']}
"""
response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
)
# Response is string not json
print(response.choices[0].message.content)

{'name': 'science fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


This will give ERROR because object properties in outupt are in single quotes

In [4]:
import json
system_prompt = """
Extract the event information from the users message and provide output in
exact json format as follows -
{'name': 'Dinner', 'date': 'Monday', 'participants': ['Michael', 'David']}
"""
response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
)
print(response.choices[0].message.content)
# This will give ERROR because object properties in outupt are in single quotes
dictionary = json.loads(response.choices[0].message.content)
print(dictionary)

{'name': 'science fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

After JSON parsing, you will receive a JSON key-value pair dictionary, not an object.

In [5]:
system_prompt = """
Extract the event information from the users message and provide output in
exact json format as follows -
{"name": "Dinner", "date": "Monday", "participants": ["Michael", "David"]}
"""
response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": "Leonardo, Ivan and Alex will be joining Taylor for dinner on Tuesday night."},
    ],
)
print(response.choices[0].message.content)
dictionary = json.loads(response.choices[0].message.content) # Convert string to json
print(dictionary)
print(dictionary["name"])
# print(dictionary.name) # Will not work because dictionary is not an object

{"name": "Dinner", "date": "Tuesday night", "participants": ["Leonardo", "Ivan", "Alex", "Taylor"]}
{'name': 'Dinner', 'date': 'Tuesday night', 'participants': ['Leonardo', 'Ivan', 'Alex', 'Taylor']}
Dinner


# Responses API

https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses

In [6]:
system_prompt = """
Extract the event information from the users message and provide output in
exact json format as follows -
{"name": "Dinner", "date": "Monday", "participants": ["Michael", "David"]}
"""
response = openai.responses.create(
    model=MODEL,
    input=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ]
)

# Response is string not json
print(response.output_text)
dictionary = json.loads(response.output_text)
print(dictionary["name"])
# print(dictionary.name) # Will not work because dictionary is not an object

{"name":"science fair","date":"Friday","participants":["Alice","Bob"]}
science fair


A similar example, except for the user input.

In [7]:
system_prompt = """
Extract the event information from the users message and provide output in
exact json format as follows -
{"name": "Dinner", "date": "Monday", "participants": ["Michael", "David"]}
"""
response = openai.responses.create(
    model=MODEL,
    input=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": "Leonardo, Ivan and Alex will be joining Taylor for dinner on Tuesday night."},
    ]
)
print(response.output_text)
dictionary = json.loads(response.output_text)
print(dictionary["name"])

{"name":"Dinner","date":"Tuesday night","participants":["Leonardo","Ivan","Alex","Taylor"]}
Dinner
